# Q1 Five-Epoch Onset Zoom

High-resolution onset comparison faithful to the modular-addition rebuttal code. Accuracy, norm, and attention diagnostics are logged every 5 epochs. Ripser/topology diagnostics are disabled. Run the cells from top to bottom.

**Colab:** mount Drive if desired, install `ripser`, then change `SAVE_ROOT` in the settings cell.


In [ ]:
# Ripser is not needed for this onset-only experiment.


In [ ]:
# Optional Google Drive mount
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
"""
Q1 temporal mechanism experiment for modular addition.

This is deliberately close to the original rebuttal code:

  * same one-layer Transformer architecture
  * same dataset construction
  * same AdamW optimizer and deterministic seeds
  * same raw/bounded output heads
  * same LayerNorm/spherical residual choices
  * JSON logs and interruption-safe recovery checkpoints

New diagnostics:

  * parameter, residual, unembedding, and logit norms
  * attention entropy and distance from uniform
  * Fourier concentration of learned representations
  * optional persistent H1 statistics with ripser

In Google Colab, install ripser once before running:

    !pip -q install ripser

The main topology comparison is:

    LayerNorm + bounded head + WD=1
    Sphere     + bounded head + WD=1

This holds the output head, temperature, optimizer, learning rate, dataset,
and weight decay fixed. The remaining configurations are component controls.
"""


In [ ]:
# %%
# Imports and editable settings

import json
import math
import os
import random
import time
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

try:
    from ripser import ripser
except ImportError:
    ripser = None


P = 113
FRAC_TRAIN = 0.30
D_MODEL = 128
NUM_HEADS = 4
MLP_DIM = 512

LR = 6e-4
SEEDS = [1, 2, 3]
MAX_EPOCHS = 20_000
LOG_EVERY = 5
TOPOLOGY_EVERY = 200
ENABLE_TOPOLOGY = False
RECOVERY_EVERY = 500
POST_99_GRACE = 1_000
STABLE_100_LOGS = 20  # 100 epochs at five-epoch logging resolution

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Change this in Colab if you want the files on Google Drive, for example:
# SAVE_ROOT = "/content/drive/MyDrive/grokking_q1_temporal"
SAVE_ROOT = "grokking_q1_onset_zoom_5epoch"
os.makedirs(SAVE_ROOT, exist_ok=True)


# Six simple, interpretable configurations.
CONFIGS = [
    # Complete WD=1 residual x head comparison.
    dict(
        name="q1_layernorm_raw_wd1",
        model_type="layernorm",
        bounded_head=False,
        tau=None,
        weight_decay=1.0,
    ),
    dict(
        name="q1_layernorm_bounded_wd1",
        model_type="layernorm",
        bounded_head=True,
        tau=10.0,
        weight_decay=1.0,
    ),
    dict(
        name="q1_sphere_raw_wd1",
        model_type="spherical",
        bounded_head=False,
        tau=None,
        weight_decay=1.0,
    ),
    dict(
        name="q1_sphere_bounded_wd1",
        model_type="spherical",
        bounded_head=True,
        tau=10.0,
        weight_decay=1.0,
    ),

    # Direct residual-geometry comparison without weight decay.
    dict(
        name="q1_layernorm_bounded_wd0",
        model_type="layernorm",
        bounded_head=True,
        tau=10.0,
        weight_decay=0.0,
    ),
    dict(
        name="q1_sphere_bounded_wd0",
        model_type="spherical",
        bounded_head=True,
        tau=10.0,
        weight_decay=0.0,
    ),
]

# Onset-only zoom: matched LayerNorm control and both bounded-sphere WD settings.
ZOOM_CONFIG_NAMES = {
    "q1_layernorm_bounded_wd1",
    "q1_sphere_bounded_wd1",
    "q1_sphere_bounded_wd0",
}
CONFIGS = [cfg for cfg in CONFIGS if cfg["name"] in ZOOM_CONFIG_NAMES]

for cfg in CONFIGS:
    cfg.update(
        p=P,
        frac_train=FRAC_TRAIN,
        d_model=D_MODEL,
        n_heads=NUM_HEADS,
        mlp_dim=MLP_DIM,
        lr=LR,
        max_epochs=MAX_EPOCHS,
        seeds=SEEDS,
    )



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# %%
# Reproducibility and data

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def make_dataset(p, frac_train, seed):
    """Exactly the same modular-addition split style as the rebuttal code."""
    set_seed(seed)

    pairs = [(a, b) for a in range(p) for b in range(p)]
    random.Random(seed).shuffle(pairs)
    n_train = int(len(pairs) * frac_train)

    def build(part):
        x = torch.tensor([[a, b, p] for a, b in part], dtype=torch.long)
        y = torch.tensor([(a + b) % p for a, b in part], dtype=torch.long)
        return x, y

    train_x, train_y = build(pairs[:n_train])
    test_x, test_y = build(pairs[n_train:])

    # The union is the complete p x p addition table. It is used only for
    # representation diagnostics, never for optimization.
    all_x = torch.cat([train_x, test_x], dim=0)
    all_y = torch.cat([train_y, test_y], dim=0)

    return train_x, train_y, test_x, test_y, all_x, all_y



In [ ]:
# %%
# Model: faithful to the previous AdditionTransformer

class AdditionTransformer(nn.Module):
    def __init__(self, p, model_type, bounded_head, tau):
        super().__init__()

        self.p = p
        self.model_type = model_type
        self.bounded_head = bounded_head
        self.tau = tau
        self.d_head = D_MODEL // NUM_HEADS

        self.tok_embed = nn.Embedding(p + 1, D_MODEL)
        self.pos_embed = nn.Embedding(3, D_MODEL)

        self.W_Q = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_K = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_V = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_O = nn.Linear(D_MODEL, D_MODEL, bias=False)

        self.mlp_in = nn.Linear(D_MODEL, MLP_DIM)
        self.mlp_out = nn.Linear(MLP_DIM, D_MODEL)
        self.unembed = nn.Linear(D_MODEL, p, bias=False)

        if model_type == "layernorm":
            self.input_norm = nn.LayerNorm(D_MODEL)
            self.norm1 = nn.LayerNorm(D_MODEL)
            self.norm2 = nn.LayerNorm(D_MODEL)

    def norm(self, h, which):
        if self.model_type == "spherical":
            return F.normalize(h, dim=-1)

        if which == 0:
            return self.input_norm(h)
        if which == 1:
            return self.norm1(h)
        return self.norm2(h)

    def forward(self, x, return_diag=False, return_hidden=False):
        _, seq_len = x.shape
        pos = torch.arange(seq_len, device=x.device).unsqueeze(0)

        h_embed_raw = self.tok_embed(x) + self.pos_embed(pos)
        h0 = self.norm(h_embed_raw, 0)

        Q = self.W_Q(h0)
        K = self.W_K(h0)
        V = self.W_V(h0)

        Q = Q.view(-1, seq_len, NUM_HEADS, self.d_head).transpose(1, 2)
        K = K.view(-1, seq_len, NUM_HEADS, self.d_head).transpose(1, 2)
        V = V.view(-1, seq_len, NUM_HEADS, self.d_head).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = F.softmax(scores, dim=-1)

        joined = (attn @ V).transpose(1, 2).contiguous()
        joined = joined.view(-1, seq_len, D_MODEL)
        attn_update = self.W_O(joined)

        attn_pre = h0 + attn_update
        h1 = self.norm(attn_pre, 1)

        mlp_update = self.mlp_out(F.relu(self.mlp_in(h1)))
        mlp_pre = h1 + mlp_update
        h2 = self.norm(mlp_pre, 2)
        h_final = h2[:, 2, :]

        if self.bounded_head:
            normalized_head = F.normalize(self.unembed.weight, dim=1)
            logits = F.linear(h_final, normalized_head) * float(self.tau)
        else:
            logits = self.unembed(h_final)

        if not return_diag and not return_hidden:
            return logits

        result = [logits]

        if return_diag:
            eps = 1e-12
            uniform = torch.full_like(attn, 1.0 / seq_len)

            entropy = -(
                attn.clamp_min(eps) * attn.clamp_min(eps).log()
            ).sum(dim=-1)
            normalized_entropy = entropy / math.log(seq_len)

            distance = (
                (attn - uniform)
                .pow(2)
                .sum(dim=-1)
                .sqrt()
            )

            # The final "=" query is the prediction-relevant query.
            equals_entropy = normalized_entropy[:, :, 2].mean()
            equals_distance = distance[:, :, 2].mean()

            attn_update_ratio = (
                attn_update.norm(dim=-1)
                / h0.norm(dim=-1).clamp_min(eps)
            )
            mlp_update_ratio = (
                mlp_update.norm(dim=-1)
                / h1.norm(dim=-1).clamp_min(eps)
            )

            attn_cosine = F.cosine_similarity(h0, h1, dim=-1)
            mlp_cosine = F.cosine_similarity(h1, h2, dim=-1)

            diag = {
                "embed_raw_norm": h_embed_raw.norm(dim=-1).mean().item(),
                "input_post_norm": h0.norm(dim=-1).mean().item(),
                "attention_update_norm": attn_update.norm(dim=-1).mean().item(),
                "attention_update_ratio": attn_update_ratio.mean().item(),
                "attn_residual_pre_norm": attn_pre.norm(dim=-1).mean().item(),
                "attn_residual_post_norm": h1.norm(dim=-1).mean().item(),
                "attention_angular_change": (
                    torch.acos(attn_cosine.clamp(-1.0, 1.0)).mean().item()
                ),
                "mlp_update_norm": mlp_update.norm(dim=-1).mean().item(),
                "mlp_update_ratio": mlp_update_ratio.mean().item(),
                "mlp_residual_pre_norm": mlp_pre.norm(dim=-1).mean().item(),
                "mlp_residual_post_norm": h2.norm(dim=-1).mean().item(),
                "mlp_angular_change": (
                    torch.acos(mlp_cosine.clamp(-1.0, 1.0)).mean().item()
                ),
                "final_hidden_norm": h_final.norm(dim=-1).mean().item(),
                "logit_l2_norm": logits.norm(dim=-1).mean().item(),
                "max_abs_logit": logits.abs().amax(dim=-1).mean().item(),
                "attention_entropy": normalized_entropy.mean().item(),
                "attention_distance_to_uniform": distance.mean().item(),
                "equals_attention_entropy": equals_entropy.item(),
                "equals_attention_distance_to_uniform": equals_distance.item(),
                "unembed_row_norm_mean": (
                    self.unembed.weight.norm(dim=1).mean().item()
                ),
                "unembed_row_norm_std": (
                    self.unembed.weight.norm(dim=1).std().item()
                ),
            }
            result.append(diag)

        if return_hidden:
            result.append(h_final)

        return tuple(result)



In [ ]:
# %%
# Diagnostic helpers

def parameter_norms(model):
    """
    Parameter norms by component.

    Important fix relative to the earlier diagnostic code:
    "unembed" must be checked before "embed", because the former contains
    the latter as a substring.
    """
    groups = {
        key: []
        for key in [
            "global",
            "embeddings",
            "attention",
            "mlp",
            "normalization",
            "unembedding",
        ]
    }

    for name, parameter in model.named_parameters():
        squared_norm = parameter.detach().float().pow(2).sum()
        groups["global"].append(squared_norm)

        if "unembed" in name:
            groups["unembedding"].append(squared_norm)
        elif "embed" in name:
            groups["embeddings"].append(squared_norm)
        elif any(key in name for key in ["W_Q", "W_K", "W_V", "W_O"]):
            groups["attention"].append(squared_norm)
        elif "mlp_" in name:
            groups["mlp"].append(squared_norm)
        elif "norm" in name:
            groups["normalization"].append(squared_norm)

    output = {}
    for group_name, values in groups.items():
        if values:
            output[f"param_norm_{group_name}"] = math.sqrt(
                torch.stack(values).sum().item()
            )
        else:
            output[f"param_norm_{group_name}"] = 0.0

    return output


def classification_margin(logits, labels):
    correct = logits.gather(1, labels[:, None]).squeeze(1)

    masked = logits.clone()
    masked.scatter_(1, labels[:, None], float("-inf"))
    largest_incorrect = masked.max(dim=1).values

    return (correct - largest_incorrect).mean().item()


def make_class_prototypes(model, all_x, all_y, p, batch_size=4096):
    """
    Average the final hidden states over all pairs with the same output class.

    The output is a p x d_model point cloud, ordered by modular class.
    """
    hidden_parts = []

    model.eval()
    with torch.no_grad():
        for start in range(0, len(all_x), batch_size):
            x_batch = all_x[start : start + batch_size].to(DEVICE)
            _, hidden = model(x_batch, return_hidden=True)
            hidden_parts.append(hidden.cpu())

    hidden = torch.cat(hidden_parts, dim=0)
    labels = all_y.cpu()

    prototypes = []
    for class_index in range(p):
        prototypes.append(hidden[labels == class_index].mean(dim=0))

    return torch.stack(prototypes, dim=0)


def top_frequency_concentration(points, top_k=5):
    """
    Simple debugging metric, not a replacement for the paper's full FVE.

    Points must be ordered by modular class. We center the representations,
    take an FFT along the class axis, discard the DC component, and report
    the fraction of spectral power in the strongest top_k frequencies.
    """
    points = points.detach().float().cpu()
    points = points - points.mean(dim=0, keepdim=True)

    spectrum = torch.fft.rfft(points, dim=0)
    power = spectrum.abs().pow(2).sum(dim=1)

    if len(power) > 0:
        power[0] = 0.0

    total = power.sum().item()
    if total <= 0.0:
        return 0.0

    k = min(top_k, len(power))
    return torch.topk(power, k=k).values.sum().item() / total


def angular_distance_matrix(points):
    points = F.normalize(points.detach().float(), dim=-1)
    similarity = (points @ points.T).clamp(-1.0, 1.0)
    distance = torch.acos(similarity) / math.pi
    return distance.cpu().numpy()


def cyclic_distance_correlation(points):
    """
    Spearman-style correlation between representation distance and the
    true cyclic distance between modular labels.
    """
    representation_distance = angular_distance_matrix(points)
    p = representation_distance.shape[0]

    indices = np.arange(p)
    raw_distance = np.abs(indices[:, None] - indices[None, :])
    cyclic_distance = np.minimum(raw_distance, p - raw_distance)

    upper = np.triu_indices(p, k=1)
    x = representation_distance[upper]
    y = cyclic_distance[upper].astype(float)

    # pandas rank handles the many ties in cyclic distance.
    x_rank = pd.Series(x).rank(method="average").to_numpy()
    y_rank = pd.Series(y).rank(method="average").to_numpy()

    if np.std(x_rank) == 0.0 or np.std(y_rank) == 0.0:
        return 0.0

    return float(np.corrcoef(x_rank, y_rank)[0, 1])


def persistent_h1_statistics(points, threshold=0.05):
    """
    Persistent H1 statistics for a p-point representation cloud.

    Returns NaNs if ripser is unavailable, so the rest of the experiment
    can still run.
    """
    if ripser is None:
        return {
            "h1_max_lifetime": float("nan"),
            "h1_total_lifetime": float("nan"),
            "h1_significant_count": float("nan"),
            "h1_birth": float("nan"),
            "h1_death": float("nan"),
            "h1_diagram": [],
        }

    distance = angular_distance_matrix(points)
    result = ripser(
        distance,
        distance_matrix=True,
        maxdim=1,
        coeff=2,
    )

    h1 = result["dgms"][1]
    finite = h1[np.isfinite(h1[:, 1])]

    if len(finite) == 0:
        return {
            "h1_max_lifetime": 0.0,
            "h1_total_lifetime": 0.0,
            "h1_significant_count": 0,
            "h1_birth": 0.0,
            "h1_death": 0.0,
            "h1_diagram": [],
        }

    lifetimes = finite[:, 1] - finite[:, 0]
    longest_index = int(np.argmax(lifetimes))

    return {
        "h1_max_lifetime": float(lifetimes[longest_index]),
        "h1_total_lifetime": float(lifetimes.sum()),
        "h1_significant_count": int((lifetimes >= threshold).sum()),
        "h1_birth": float(finite[longest_index, 0]),
        "h1_death": float(finite[longest_index, 1]),
        "h1_diagram": finite.tolist(),
    }


def topology_diagnostics(model, all_x, all_y, p):
    prototypes = make_class_prototypes(model, all_x, all_y, p)

    token_points = model.tok_embed.weight[:p].detach()
    unembedding_points = model.unembed.weight.detach()

    output = {}

    # Primary topology: final-hidden output-class prototypes.
    h1 = persistent_h1_statistics(prototypes)
    output.update({f"prototype_{key}": value for key, value in h1.items()})
    output["prototype_top5_fourier_power"] = top_frequency_concentration(
        prototypes, top_k=5
    )
    output["prototype_cyclic_distance_correlation"] = (
        cyclic_distance_correlation(prototypes)
    )

    # Secondary, cheap representation diagnostics.
    output["token_top5_fourier_power"] = top_frequency_concentration(
        token_points, top_k=5
    )
    output["unembedding_top5_fourier_power"] = top_frequency_concentration(
        unembedding_points, top_k=5
    )

    output["token_h1_max_lifetime"] = persistent_h1_statistics(
        token_points
    )["h1_max_lifetime"]
    output["unembedding_h1_max_lifetime"] = persistent_h1_statistics(
        unembedding_points
    )["h1_max_lifetime"]

    return output



In [ ]:
# %%
# Training

def first_onset(history, key, threshold):
    for epoch, value in zip(history["epochs"], history[key]):
        if value >= threshold:
            return epoch
    return None


def first_sustained_test_rise(history):
    """
    Same qualitative definition used in the rebuttal:

    first logged epoch with test accuracy >10% followed by a gain of at
    least five percentage points within the next 100 epochs.
    """
    epochs = history["epochs"]
    values = history["test_acc"]

    for index, (epoch, value) in enumerate(zip(epochs, values)):
        if value <= 0.10:
            continue

        future_limit = epoch + 100
        future_values = [
            later_value
            for later_epoch, later_value in zip(epochs[index + 1 :], values[index + 1 :])
            if later_epoch <= future_limit
        ]

        if future_values and max(future_values) >= value + 0.05:
            return epoch

    return None


def train_one(model, data, cfg, recovery_path):
    train_x, train_y, test_x, test_y, all_x, all_y = data

    train_x = train_x.to(DEVICE)
    train_y = train_y.to(DEVICE)
    test_x = test_x.to(DEVICE)
    test_y = test_y.to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
        betas=(0.9, 0.999),
    )

    history = {
        key: []
        for key in [
            "epochs",
            "train_acc",
            "test_acc",
            "train_loss",
            "test_loss",
        ]
    }
    diagnostics = []

    start_epoch = 0
    test99_epoch = None
    stable100 = 0
    stop_reason = "max_epochs"

    if os.path.exists(recovery_path):
        state = torch.load(recovery_path, map_location=DEVICE)
        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        history = state["history"]
        diagnostics = state["diagnostics"]
        start_epoch = state["epoch"] + 1
        test99_epoch = state["test99_epoch"]
        stable100 = state["stable100"]
        print("resume:", cfg["name"], "from", start_epoch)

    iterator = tqdm(
        range(start_epoch, cfg["max_epochs"]),
        desc=cfg["name"],
        leave=False,
    )

    for epoch in iterator:
        model.train()

        loss = criterion(model(train_x), train_y)
        if not torch.isfinite(loss):
            stop_reason = f"non_finite_loss_epoch_{epoch}"
            break

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        should_log = (
            epoch % LOG_EVERY == 0
            or epoch == cfg["max_epochs"] - 1
        )
        if not should_log:
            continue

        model.eval()
        with torch.no_grad():
            train_logits = model(train_x)
            test_logits, diag = model(test_x, return_diag=True)

            train_loss = criterion(train_logits, train_y).item()
            test_loss = criterion(test_logits, test_y).item()
            train_acc = (
                (train_logits.argmax(dim=-1) == train_y)
                .float()
                .mean()
                .item()
            )
            test_acc = (
                (test_logits.argmax(dim=-1) == test_y)
                .float()
                .mean()
                .item()
            )

            diag["classification_margin"] = classification_margin(
                test_logits, test_y
            )
            diag.update(parameter_norms(model))
            diag["epoch"] = epoch

        if ENABLE_TOPOLOGY and epoch % TOPOLOGY_EVERY == 0:
            topology = topology_diagnostics(
                model=model,
                all_x=all_x,
                all_y=all_y,
                p=cfg["p"],
            )
            diag.update(topology)

        diagnostics.append(diag)

        for key, value in [
            ("epochs", epoch),
            ("train_acc", train_acc),
            ("test_acc", test_acc),
            ("train_loss", train_loss),
            ("test_loss", test_loss),
        ]:
            history[key].append(value)

        iterator.set_postfix(
            train=f"{100 * train_acc:.1f}%",
            test=f"{100 * test_acc:.1f}%",
        )

        if test99_epoch is None and test_acc >= 0.99:
            test99_epoch = epoch

        if test_acc >= 1.0 - 1e-12:
            stable100 += 1
        else:
            stable100 = 0

        if epoch % RECOVERY_EVERY == 0:
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "history": history,
                    "diagnostics": diagnostics,
                    "test99_epoch": test99_epoch,
                    "stable100": stable100,
                },
                recovery_path,
            )

        if stable100 >= STABLE_100_LOGS:
            stop_reason = "stable_test_100"
            break

        if (
            test99_epoch is not None
            and epoch >= test99_epoch + POST_99_GRACE
        ):
            stop_reason = "post_99_grace_complete"
            break

    return history, diagnostics, stop_reason


def make_summary(cfg, seed, history, stop_reason, elapsed_seconds):
    summary = {
        "config": cfg["name"],
        "seed": seed,
        "model_type": cfg["model_type"],
        "bounded_head": cfg["bounded_head"],
        "tau": cfg["tau"],
        "weight_decay": cfg["weight_decay"],
        "lr": cfg["lr"],
        "max_epochs": cfg["max_epochs"],
        "stop_reason": stop_reason,
        "elapsed_seconds": elapsed_seconds,
        "last_epoch": history["epochs"][-1] if history["epochs"] else None,
        "peak_test_acc": (
            max(history["test_acc"]) if history["test_acc"] else None
        ),
        "first_sustained_test_rise": first_sustained_test_rise(history),
    }

    for label, threshold in [
        ("95", 0.95),
        ("99", 0.99),
        ("100", 1.0 - 1e-12),
    ]:
        train_epoch = first_onset(history, "train_acc", threshold)
        test_epoch = first_onset(history, "test_acc", threshold)

        summary[f"train_epoch_{label}"] = train_epoch
        summary[f"test_epoch_{label}"] = test_epoch
        summary[f"gap_{label}"] = (
            None
            if train_epoch is None or test_epoch is None
            else test_epoch - train_epoch
        )

    return summary


def run_all_experiments():
    rows = []

    if ENABLE_TOPOLOGY and ripser is None:
        print(
            "\nWARNING: ripser is not installed. Training will run, but H1 "
            "statistics will be NaN. In Colab run: !pip -q install ripser\n"
        )

    for cfg_template in CONFIGS:
        cfg = deepcopy(cfg_template)
        cfg_dir = os.path.join(SAVE_ROOT, cfg["name"])
        os.makedirs(cfg_dir, exist_ok=True)

        for seed in cfg["seeds"]:
            stem = f"{cfg['name']}_seed{seed}"
            log_path = os.path.join(cfg_dir, stem + ".json")
            model_path = os.path.join(cfg_dir, stem + ".pt")
            recovery_path = os.path.join(cfg_dir, stem + "_recovery.pt")

            if os.path.exists(log_path) and os.path.exists(model_path):
                with open(log_path, "r") as handle:
                    saved = json.load(handle)
                rows.append(saved["summary"])
                print("skip:", stem)
                continue

            print("\nstarting:", stem)
            data = make_dataset(cfg["p"], cfg["frac_train"], seed)

            set_seed(seed)
            model = AdditionTransformer(
                p=cfg["p"],
                model_type=cfg["model_type"],
                bounded_head=cfg["bounded_head"],
                tau=cfg["tau"],
            ).to(DEVICE)

            start_time = time.time()
            history, diagnostics, stop_reason = train_one(
                model=model,
                data=data,
                cfg=cfg,
                recovery_path=recovery_path,
            )
            elapsed_seconds = time.time() - start_time

            summary = make_summary(
                cfg=cfg,
                seed=seed,
                history=history,
                stop_reason=stop_reason,
                elapsed_seconds=elapsed_seconds,
            )

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "summary": summary,
                    "config": cfg,
                },
                model_path,
            )

            with open(log_path, "w") as handle:
                json.dump(
                    {
                        "summary": summary,
                        "history": history,
                        "diagnostics": diagnostics,
                    },
                    handle,
                    indent=2,
                )

            if os.path.exists(recovery_path):
                os.remove(recovery_path)

            rows.append(summary)
            pd.DataFrame(rows).to_csv(
                os.path.join(SAVE_ROOT, "summary_incremental.csv"),
                index=False,
            )

            print("saved:", stem)
            print(summary)

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    summary_frame = pd.DataFrame(rows)
    summary_frame.to_csv(
        os.path.join(SAVE_ROOT, "summary_final.csv"),
        index=False,
    )

    print("\nPer-run results:")
    print(
        summary_frame
        .sort_values(["config", "seed"])
        .to_string(index=False)
    )

    return summary_frame



In [ ]:
# %%
# Analysis and simple figures

def load_run(config_name, seed):
    path = os.path.join(
        SAVE_ROOT,
        config_name,
        f"{config_name}_seed{seed}.json",
    )
    with open(path, "r") as handle:
        return json.load(handle)


def diagnostic_frame(saved_run):
    frame = pd.DataFrame(saved_run["diagnostics"])
    return frame.sort_values("epoch").reset_index(drop=True)


def history_frame(saved_run):
    return pd.DataFrame(saved_run["history"])


def nearest_topology_row(frame, target_epoch):
    available = frame[
        frame["prototype_top5_fourier_power"].notna()
    ].copy()

    if len(available) == 0:
        return None

    index = (available["epoch"] - target_epoch).abs().idxmin()
    return available.loc[index]


def make_main_pair_figure(seed=1):
    """
    Clean residual-geometry comparison:

        LayerNorm + bounded + WD=1
        Sphere     + bounded + WD=1
    """
    names = [
        "q1_layernorm_bounded_wd1",
        "q1_sphere_bounded_wd1",
    ]
    labels = ["LayerNorm + bounded", "Sphere + bounded"]
    colors = ["tab:blue", "tab:orange"]

    runs = [load_run(name, seed) for name in names]
    histories = [history_frame(run) for run in runs]
    diagnostics = [diagnostic_frame(run) for run in runs]

    figure, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    for label, color, history, diag in zip(
        labels, colors, histories, diagnostics
    ):
        axes[0].plot(
            history["epochs"],
            history["train_acc"],
            linestyle="--",
            color=color,
            alpha=0.7,
            label=f"{label}: train",
        )
        axes[0].plot(
            history["epochs"],
            history["test_acc"],
            color=color,
            linewidth=2,
            label=f"{label}: test",
        )

        topology = diag[
            diag["prototype_h1_max_lifetime"].notna()
        ]
        axes[1].plot(
            topology["epoch"],
            topology["prototype_h1_max_lifetime"],
            marker="o",
            markersize=3,
            color=color,
            label=label,
        )
        axes[2].plot(
            topology["epoch"],
            topology["prototype_top5_fourier_power"],
            marker="o",
            markersize=3,
            color=color,
            label=label,
        )

    axes[0].set_title(f"Accuracy, seed {seed}")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_ylim(-0.02, 1.02)

    axes[1].set_title("Longest persistent H1 lifetime")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Lifetime")

    axes[2].set_title("Prototype top-5 Fourier power")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Fraction of non-DC power")

    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend(fontsize=8)

    figure.tight_layout()
    output_path = os.path.join(
        SAVE_ROOT,
        f"main_pair_accuracy_topology_seed{seed}.png",
    )
    figure.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()

    return output_path


def make_main_pair_mechanism_figure(seed=1):
    names = [
        "q1_layernorm_bounded_wd1",
        "q1_sphere_bounded_wd1",
    ]
    labels = ["LayerNorm + bounded", "Sphere + bounded"]
    colors = ["tab:blue", "tab:orange"]

    diagnostics = [
        diagnostic_frame(load_run(name, seed))
        for name in names
    ]

    figure, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.ravel()

    metrics = [
        ("param_norm_global", "Global parameter norm"),
        ("unembed_row_norm_mean", "Mean unembedding row norm"),
        ("attn_residual_pre_norm", "Pre-normalization attention residual"),
        ("mlp_residual_pre_norm", "Pre-normalization MLP residual"),
        ("logit_l2_norm", "Mean logit L2 norm"),
        ("equals_attention_entropy", "Equals-query normalized entropy"),
    ]

    for axis, (metric, title) in zip(axes, metrics):
        for label, color, frame in zip(labels, colors, diagnostics):
            axis.plot(
                frame["epoch"],
                frame[metric],
                color=color,
                linewidth=2,
                label=label,
            )

        axis.set_title(title)
        axis.set_xlabel("Epoch")
        axis.grid(alpha=0.25)
        axis.legend(fontsize=8)

    figure.tight_layout()
    output_path = os.path.join(
        SAVE_ROOT,
        f"main_pair_mechanism_seed{seed}.png",
    )
    figure.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()

    return output_path


def make_snapshot_table(seed=1):
    """
    Three paired checkpoints:

      A. first epoch at which both models have Train-99
      B. sphere Test-99 epoch
      C. LayerNorm Test-99 epoch

    Both models are evaluated at the same absolute epoch within each snapshot.
    """
    ln_name = "q1_layernorm_bounded_wd1"
    sphere_name = "q1_sphere_bounded_wd1"

    ln_run = load_run(ln_name, seed)
    sphere_run = load_run(sphere_name, seed)

    ln_summary = ln_run["summary"]
    sphere_summary = sphere_run["summary"]

    both_memorized = max(
        ln_summary["train_epoch_99"],
        sphere_summary["train_epoch_99"],
    )
    sphere_generalized = sphere_summary["test_epoch_99"]
    layernorm_generalized = ln_summary["test_epoch_99"]

    targets = {
        "A_both_train99": both_memorized,
        "B_sphere_test99": sphere_generalized,
        "C_layernorm_test99": layernorm_generalized,
    }

    rows = []
    for snapshot_name, target_epoch in targets.items():
        for model_name, saved_run in [
            (ln_name, ln_run),
            (sphere_name, sphere_run),
        ]:
            history = history_frame(saved_run)
            diag = diagnostic_frame(saved_run)

            history_index = (
                history["epochs"] - target_epoch
            ).abs().idxmin()
            history_row = history.loc[history_index]
            topology_row = nearest_topology_row(diag, target_epoch)

            row = {
                "seed": seed,
                "snapshot": snapshot_name,
                "target_epoch": target_epoch,
                "model": model_name,
                "observed_epoch": int(history_row["epochs"]),
                "train_acc": float(history_row["train_acc"]),
                "test_acc": float(history_row["test_acc"]),
            }

            if topology_row is not None:
                row.update(
                    topology_epoch=int(topology_row["epoch"]),
                    prototype_h1_max_lifetime=float(
                        topology_row["prototype_h1_max_lifetime"]
                    ),
                    prototype_top5_fourier_power=float(
                        topology_row["prototype_top5_fourier_power"]
                    ),
                    prototype_cyclic_distance_correlation=float(
                        topology_row[
                            "prototype_cyclic_distance_correlation"
                        ]
                    ),
                )

            rows.append(row)

    frame = pd.DataFrame(rows)
    output_path = os.path.join(
        SAVE_ROOT,
        f"snapshot_summary_seed{seed}.csv",
    )
    frame.to_csv(output_path, index=False)

    print("\nThree-checkpoint comparison:")
    print(frame.to_string(index=False))

    return frame


def make_snapshot_persistence_figure(seed=1):
    """
    Persistence diagrams for the same three paired checkpoints used by
    make_snapshot_table().

    Rows are models and columns are matched moments in training.
    """
    if ripser is None:
        print("Skipping persistence diagrams because ripser is unavailable.")
        return None

    ln_name = "q1_layernorm_bounded_wd1"
    sphere_name = "q1_sphere_bounded_wd1"

    ln_run = load_run(ln_name, seed)
    sphere_run = load_run(sphere_name, seed)

    ln_summary = ln_run["summary"]
    sphere_summary = sphere_run["summary"]

    targets = [
        (
            "A: both Train-99",
            max(
                ln_summary["train_epoch_99"],
                sphere_summary["train_epoch_99"],
            ),
        ),
        ("B: sphere Test-99", sphere_summary["test_epoch_99"]),
        ("C: LayerNorm Test-99", ln_summary["test_epoch_99"]),
    ]

    model_rows = [
        ("LayerNorm + bounded", ln_run),
        ("Sphere + bounded", sphere_run),
    ]

    figure, axes = plt.subplots(2, 3, figsize=(13, 8))

    for row_index, (model_label, saved_run) in enumerate(model_rows):
        diag = diagnostic_frame(saved_run)

        for column_index, (snapshot_label, target_epoch) in enumerate(targets):
            axis = axes[row_index, column_index]
            topology_row = nearest_topology_row(diag, target_epoch)

            if topology_row is None:
                axis.text(
                    0.5,
                    0.5,
                    "No topology record",
                    ha="center",
                    va="center",
                )
                axis.set_axis_off()
                continue

            diagram = np.asarray(
                topology_row["prototype_h1_diagram"],
                dtype=float,
            )

            if diagram.size == 0:
                diagram = diagram.reshape(0, 2)

            maximum = 1.0
            if len(diagram):
                maximum = max(maximum, float(diagram.max()) * 1.05)
                axis.scatter(
                    diagram[:, 0],
                    diagram[:, 1],
                    s=22,
                    alpha=0.75,
                    color="tab:purple",
                )

            axis.plot(
                [0.0, maximum],
                [0.0, maximum],
                linestyle="--",
                color="black",
                alpha=0.5,
            )
            axis.set_xlim(0.0, maximum)
            axis.set_ylim(0.0, maximum)
            axis.set_xlabel("Birth")
            axis.set_ylabel("Death")
            axis.grid(alpha=0.2)

            observed_epoch = int(topology_row["epoch"])
            longest = float(topology_row["prototype_h1_max_lifetime"])

            axis.set_title(
                f"{snapshot_label}\n"
                f"epoch {observed_epoch}, max lifetime={longest:.3f}"
            )

            if column_index == 0:
                axis.text(
                    -0.35,
                    0.5,
                    model_label,
                    transform=axis.transAxes,
                    rotation=90,
                    va="center",
                    ha="center",
                    fontsize=11,
                    fontweight="bold",
                )

    figure.suptitle(
        f"Persistent H1 of output-class prototypes, seed {seed}",
        fontsize=14,
    )
    figure.tight_layout(rect=[0.03, 0.0, 1.0, 0.95])

    output_path = os.path.join(
        SAVE_ROOT,
        f"snapshot_persistence_seed{seed}.png",
    )
    figure.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()

    return output_path



In [ ]:
# %%
# Run this file

if __name__ == "__main__":
    print("Device:", DEVICE)
    print("Saving to:", os.path.abspath(SAVE_ROOT))
    print(
        f"{len(CONFIGS)} configurations x {len(SEEDS)} seeds = "
        f"{len(CONFIGS) * len(SEEDS)} runs"
    )

    run_all_experiments()

    # Compact onset table used for the camera-ready phase-ordering check.
    onset_rows = []
    for cfg in CONFIGS:
        for seed in SEEDS:
            saved = load_run(cfg["name"], seed)
            summary = saved["summary"]
            history = history_frame(saved)
            rise = summary["first_sustained_test_rise"]
            rise_row = history.loc[history["epochs"] == rise].iloc[0]
            onset_rows.append({
                "config": cfg["name"],
                "seed": seed,
                "logging_resolution": LOG_EVERY,
                "test_rise": rise,
                "train_acc_at_rise": rise_row["train_acc"],
                "test_acc_at_rise": rise_row["test_acc"],
                "train99": summary["train_epoch_99"],
                "train100": summary["train_epoch_100"],
                "rise_minus_train99": (
                    None if summary["train_epoch_99"] is None
                    else rise - summary["train_epoch_99"]
                ),
                "rise_minus_train100": (
                    None if summary["train_epoch_100"] is None
                    else rise - summary["train_epoch_100"]
                ),
            })

    onset_frame = pd.DataFrame(onset_rows)
    onset_frame.to_csv(
        os.path.join(SAVE_ROOT, "onset_zoom_5epoch.csv"),
        index=False,
    )
    print("\nFive-epoch onset comparison:")
    print(onset_frame.to_string(index=False))

    plot_labels = {
        "q1_layernorm_bounded_wd1": "LayerNorm + bounded, WD=1",
        "q1_sphere_bounded_wd1": "Sphere + bounded, WD=1",
        "q1_sphere_bounded_wd0": "Sphere + bounded, WD=0",
    }
    plot_colors = ["tab:blue", "tab:orange", "tab:green"]

    for selected_seed in SEEDS:
        figure, axes = plt.subplots(1, 2, figsize=(14, 4.5))
        for cfg, color in zip(CONFIGS, plot_colors):
            history = history_frame(load_run(cfg["name"], selected_seed))
            label = plot_labels[cfg["name"]]
            for axis in axes:
                axis.plot(
                    history["epochs"], history["train_acc"],
                    linestyle="--", color=color, alpha=0.7,
                    label=f"{label}: train",
                )
                axis.plot(
                    history["epochs"], history["test_acc"],
                    color=color, linewidth=2,
                    label=f"{label}: test",
                )

        axes[0].set_title(f"Full training, seed {selected_seed}")
        axes[1].set_title(f"First 2,000 epochs, seed {selected_seed}")
        axes[1].set_xlim(0, 2_000)
        for axis in axes:
            axis.set_xlabel("Epoch")
            axis.set_ylabel("Accuracy")
            axis.set_ylim(-0.02, 1.02)
            axis.grid(alpha=0.25)
            axis.legend(fontsize=7)
        figure.tight_layout()
        figure.savefig(
            os.path.join(SAVE_ROOT, f"onset_zoom_seed{selected_seed}.png"),
            dpi=180, bbox_inches="tight",
        )
        plt.show()
